In [96]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


## 사용 함수

In [97]:
# 컬럼 정보 간단 표현
def check_basic_info(df, df_name, exclude_cols=None):
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 컬럼 정보 / 결측치 확인 정보 요약")
    print(f"{'='*80}\n")


    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)
    
    # 1. 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })
    
    # 2. 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    })
    
    # 3. 보기 좋게 정렬
    summary_df = summary_df.sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )
    
    print("[전체 요약]")
    display(overview_df)
    
    print("[컬럼별 요약]")
    display(summary_df)

    print("[테이블 요약]")
    display(df.head())

In [98]:
# 컬럼 분포 확인
def check_category_summary(df, df_name, col_name):
    df_copied = df.copy()
    
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")
    
    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return
    
    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, '개수']
    summary_df['비율(%)'] = (summary_df['개수'] / len(df_copied) * 100).round(2)
    
    print("전체 행 수:", len(df_copied))
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()
    
    display(summary_df.head(10))

# 데이터 로드

In [99]:
# ============================================================
# 1. 원본 데이터 로드
# ============================================================
df = pd.read_csv("data/transcript_portfolio_profile.csv")

# 테이블 복제
df2 = df.copy()
check_basic_info(df2, 'transcript_portfolio')


transcript_portfolio의 컬럼 정보 / 결측치 확인 정보 요약

[전체 요약]


,항목,값
0,행 개수,306137
1,열 개수,20
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
event_reward,float64,33182,10.84,272955,89.16,4
amount,float64,138953,45.39,167184,54.61,5103
offer_id,str,167184,54.61,138953,45.39,10
offer_reward,float64,167184,54.61,138953,45.39,5
difficulty,float64,167184,54.61,138953,45.39,5
duration,float64,167184,54.61,138953,45.39,5
channels,str,167184,54.61,138953,45.39,4
offer_type,str,167184,54.61,138953,45.39,3
web,float64,167184,54.61,138953,45.39,2
mobile,float64,167184,54.61,138953,45.39,2


[테이블 요약]


,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,has_profile
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [100]:
# 이벤트별 구조적 결측깂

struct_missing_summary = (
    df2.groupby('event')[['offer_id', 'amount', 'event_reward']]
    .agg(lambda s: s.isna().sum())
)

display(struct_missing_summary)

,offer_id,amount,event_reward
event,,,
offer completed,0,33182,0
offer received,0,76277,76277
offer viewed,0,57725,57725
transaction,138953,0,138953


## 결측 해석 주의

데이터의 일부 결측은 데이터 오류가 아니라 이벤트 구조 차이에서 발생한 구조적 결측이다.

예를 들어,
- transaction 이벤트에서 offer_id가 없는 것은 정상이다.
- transaction이 아닌 이벤트에서 amount가 없는 것도 정상이다.
- offer completed가 아닌 이벤트에서 event_reward가 없는 것도 정상이다.

따라서 결측은 일괄적으로 데이터 품질 문제로 해석하지 않고,\
이벤트 구조에 따른 정상 결측과 실제 미매칭 결측을 구분해 봐야 한다.

# 전환율 테이블 생성

이 분석에선 전환율은 received를 기준으로 viewed, completed 반응을 연결해 계산한다.

- viewed, completed는 반드시 offer의 duration(유효기간) 내에서 발생한 경우만 유효 반응으로 인정한다.
- duration을 고려하지 않으면 오퍼 반응률과 완료율이 과대 또는 왜곡될 수 있다.
- transaction은 특정 offer와 직접 연결되지 않으므로, 본 분석에서는 프로모션 직접 성과 지표로써 사용하지 않는다.

시간 순서를 고려해\
고객 세그먼트, reward, difficulty, channel, offer_type 같은 변수와 붙여 분석한다.

## received / completed 이벤트 분리
전환율의 분모와 분자 생성

In [101]:
# ============================================================
# received / viewed / completed 이벤트 분리
# ============================================================
# received 분리
received = (
    df2[df2['event'] == 'offer received'][
        [
            'customer_id', 'offer_id', 'time',
            'offer_type', 'offer_reward', 'difficulty', 'duration',
            'web', 'email', 'mobile', 'social',
            'has_profile',
            'gender', 'age', 'income', 'became_member_on'
        ]
    ]
    .copy()
    .rename(columns={'time': 'time_received'})
)

# viewed 분리
viewed = (
    df2[df2['event'] == 'offer viewed'][
        ['customer_id', 'offer_id', 'time']
    ]
    .copy()
    .rename(columns={'time': 'time_viewed'})
)

# completed 분리
completed = (
    df2[df2['event'] == 'offer completed'][
        ['customer_id', 'offer_id', 'time']
    ]
    .copy()
    .rename(columns={'time': 'time_completed'})
)

print("received 행 수:", len(received))
print("viewed 행 수:", len(viewed))
print("completed 행 수:", len(completed))

received 행 수: 76277
viewed 행 수: 57725
completed 행 수: 33182


## 시간순 정렬과 반응 단위 정의
동일 고객이 동일 offer를 여러 번 받을 수 있으므로,\
(customer_id + offer_id) 조합만으로는 하나의 고유 반응 단위를 정의할 수 없다.

따라서 본 분석에서는 received 이벤트 1건을 하나의 반응 단위(instance처럼)로 보고,\
이후 발생한 viewed, completed를 시간 순서와 duration 조건에 맞게 연결한다.

원래는 seq 방식으로 n번째 received와 n번째 completed를 연결하고자 했으나,\
같은 고객이 같은 오퍼를 이전 오퍼 유효기간이 끝나기 전에 또 받은 경우\
잘못 연결될 가능성이 있어 방식 변경이 필요했다.

그래서 completed보다 먼저 발생한 received만 후보로 보고,\
그중 가장 최근의 유효한 received를 연결하는 구조로 정의하였다.

추가적으로 생성하는 시간 컬럼
- time_completed_rc : received → completed용 completed 시간
- time_completed_vc : viewed → completed용 completed 시간

플래그 컬럼
- converted_rv : received → viewed
- converted_rc : received → completed
- converted_vc : viewed → completed
- converted_rvc : received → viewed → completed


조건에 맞는 received가 없으면 해당 completed는 연결하지 않는다.\
즉, completed는 존재하더라도 유효한 received와 연결되지 않으면 분석용 match_df에는 포함되지 않는다.

In [102]:
# ============================================================
# 정렬 + 기준키 생성
# ============================================================
received = received.sort_values(
    ["customer_id", "offer_id", "time_received"]
).reset_index(drop=True)

viewed = viewed.sort_values(
    ["customer_id", "offer_id", "time_viewed"]
).reset_index(drop=True)

completed = completed.sort_values(
    ["customer_id", "offer_id", "time_completed"]
).reset_index(drop=True)

# received 기준 고유 행 번호 만들기
received["received_idx"] = received.index

# 오퍼 유효 종료 시점
received["offer_end_time"] = received["time_received"] + received["duration"] * 24

In [103]:
# ============================================================
# 공통 매칭 함수
# - event를 가장 최근의 유효한 base 이벤트에 연결
# - 시간 순서 맞게
# - duration 안의 이벤트만 인정
# - 1대1방식으로 (한 base는 같은 방식에서 1번만 사용)
# - 속도 개선 버전
# ============================================================
def match_event_to_base(
        base_df,           # 기준이 되는 이벤트 테이블(received 또는 viewed)
        event_df,          # base에 연결할 이벤트 테이블(viewed 또는 completed)
        base_time_col,     # base 이벤트의 시간 컬럼명
        event_time_col,    # 연결할 event의 시간 컬럼명
        matched_time_col,  # 매칭 후 result에 새로 붙일 시간 컬럼명
        base_id_col="received_idx"  # base 행을 구분하는 고유 식별 컬럼명
        ):
    """
    received 또는 viewed를 기준으로 잡고 그 뒤에 일어난 viewed/completed를
    같은 고객과 같은 오퍼 안에서 가장 최근의 유효한 base에 1대1로 연결
    """
    
    # 매칭 결과를 저장할 리스트
    match_rows = []

    # base 이벤트(received 또는 viewed?) 고객, 오퍼, 시간 순서대로 정렬
    # -> 불필요한 컬럼을 줄여서 속도와 메모리 사용을 줄임
    base_sorted = (
        base_df[["customer_id", "offer_id", base_id_col, base_time_col, "offer_end_time"]]
        .sort_values(["customer_id", "offer_id", base_time_col])
        .reset_index(drop=True)
    )

    # event 이벤트(viewed 또는 completed) 고객, 오퍼, 시간 순서대로 정렬
    # 이벤트 발생 순서대로 base와 연결
    event_sorted = (
        event_df[["customer_id", "offer_id", event_time_col]]
        .sort_values(["customer_id", "offer_id", event_time_col])
        .reset_index(drop=True)
    )

    # base를 customer_id + offer_id 그룹별로 한 번만 저장
    # -> 반복문 안에서 base_group을 계속 다시 찾지 않기 위함
    # -> numpy 배열로 바꿔서 더 빠르게 처리
    base_groups = {
        key: group[[base_id_col, base_time_col, "offer_end_time"]].to_numpy()
        for key, group in base_sorted.groupby(["customer_id", "offer_id"], sort=False)
    }

    # event를 customer_id + offer_id 단위로 묶어서 반복
    for key, event_group in event_sorted.groupby(["customer_id", "offer_id"], sort=False):

        # 현재 그룹에 해당하는 base 가져오기
        bases = base_groups.get(key)

        # 해당 고객+오퍼의 base가 아예 없으면 다음 그룹으로
        if bases is None:
            continue

        # numpy 배열에서 필요한 값 분리
        # base_ids        : base 행의 고유 식별값
        # base_times      : base 발생 시간
        # offer_end_times : 각 base의 유효 종료 시점
        base_ids = bases[:, 0]
        base_times = bases[:, 1].astype(float)
        offer_end_times = bases[:, 2].astype(float)

        # 같은 방식에서 한 base는 1번만 사용
        # -> 이미 사용한 base는 True로 바뀜
        used = np.zeros(len(bases), dtype=bool)

        # event 발생 시간을 하나씩 확인하면서 조건에 맞는 base 찾기
        for event_time in event_group[event_time_col].to_numpy(dtype=float):

            # event_time 이하인 base 중
            # 가장 최근 base 위치를 빠르게 찾음
            # -> searchsorted는 정렬된 배열에서 위치를 빠르게 찾는 함수
            j = np.searchsorted(base_times, event_time, side="right") - 1

            # 가장 최근 후보부터 뒤에서 앞으로 확인
            # -> event와 가장 가까운 이전 base를 선택하기 위함
            while j >= 0:

                # 조건 1 아직 사용되지 않은 base인가?
                # 조건 2 event가 offer_end_time 이내인가?
                # -> duration 안에서 일어난 이벤트만 인정
                if (not used[j]) and (offer_end_times[j] >= event_time):
                    match_rows.append({
                        base_id_col: base_ids[j],
                        matched_time_col: event_time
                    })

                    # 사용한 base는 다시 못 쓰게 저장
                    used[j] = True
                    break

                # 조건이 안 맞으면 더 이전 base를 확인
                j -= 1

    # 최종 결과는 base_df를 기준으로 유지
    result = base_df.copy()

    # 매칭 결과가 하나도 없으면 결측치로 만든 뒤 반환
    if not match_rows:
        result[matched_time_col] = pd.NA
        return result

    # 매칭 결과 리스트를 DataFrame으로 변환
    match_df = pd.DataFrame(match_rows)

    # base_df와 매칭 결과를 base_id_col 기준으로 왼쪽 병합
    # -> 매칭 성공한 행은 시간값이 붙고
    # -> 실패한 행은 NaN으로 남음
    result = result.merge(
        match_df,
        on=base_id_col,
        how="left"
    )

    return result

## 4개 전환 기준 테이블 생성

In [104]:
# ============================================================
# 1) received -> viewed 매칭
# ============================================================
rv = match_event_to_base(
    base_df=received,                 # 기준: 받은 오퍼(received)
    event_df=viewed,                  # 연결 대상: 열람 이벤트(viewed)
    base_time_col="time_received",    # 기준 시간: 오퍼 받은 시점
    event_time_col="time_viewed",     # 연결할 이벤트 시간: 오퍼 본 시점
    matched_time_col="time_viewed"    # 결과로 붙일 컬럼명
)

In [105]:
# ============================================================
# 2) received -> completed 매칭
# ============================================================
rc_match = match_event_to_base(
    base_df=received,                    # 기준: 받은 오퍼(received)
    event_df=completed,                  # 연결 대상: 완료 이벤트(completed)
    base_time_col="time_received",       # 기준 시간: 오퍼 받은 시점
    event_time_col="time_completed",     # 연결할 이벤트 시간: 오퍼 완료 시점
    matched_time_col="time_completed_rc" # 결과로 붙일 컬럼명(received→completed용)
)


In [106]:
# ============================================================
# 3) viewed -> completed 매칭
# viewed가 붙은 received만 대상으로, viewed 이후 completed를 다시 매칭
# ============================================================
view_base = rv[rv["time_viewed"].notna()].copy()

vc_match = match_event_to_base(
    base_df=view_base,                  # 기준: viewed가 확인된 received
    event_df=completed,                 # 연결 대상: 완료 이벤트(completed)
    base_time_col="time_viewed",        # 기준 시간: 오퍼 본 시점
    event_time_col="time_completed",    # 연결할 이벤트 시간: 오퍼 완료 시점
    matched_time_col="time_completed_vc" # 결과로 붙일 컬럼명(viewed→completed용)
)

In [107]:
# ============================================================
# 최종 received 기준 퍼널 테이블 생성
# ============================================================
funnel = received.copy()

# received -> viewed 매칭 결과 붙이기
funnel = funnel.merge(
    rv[["received_idx", "time_viewed"]],
    on="received_idx",
    how="left"
)

# received -> completed 매칭 결과 붙이기
funnel = funnel.merge(
    rc_match[["received_idx", "time_completed_rc"]],
    on="received_idx",
    how="left"
)

# 시간 차이 계산
# 각 단계까지 걸린 시간을 계산
funnel = funnel.merge(
    vc_match[["received_idx", "time_completed_vc"]],
    on="received_idx",
    how="left"
)

# 시간 차이
funnel["rv_time_diff"] = funnel["time_viewed"] - funnel["time_received"]
funnel["rc_time_diff"] = funnel["time_completed_rc"] - funnel["time_received"]
funnel["vc_time_diff"] = funnel["time_completed_vc"] - funnel["time_viewed"]

# 유효 여부 플래그
# 시간값이 존재하면 1, 없으면 0으로 변환
funnel["has_viewed"] = funnel["time_viewed"].notna().astype(int)
funnel["has_completed_rc"] = funnel["time_completed_rc"].notna().astype(int)
funnel["has_completed_after_view"] = funnel["time_completed_vc"].notna().astype(int)

# 전환 플래그
# converted_rv  : received -> viewed 전환 여부
funnel["converted_rv"] = funnel["has_viewed"]

# converted_rc  : received -> completed 전환 여부
funnel["converted_rc"] = funnel["has_completed_rc"]

# converted_vc  : viewed -> completed 전환 여부
funnel["converted_vc"] = funnel["has_completed_after_view"]

# converted_rvc : received -> viewed -> completed 전환 여부
funnel["converted_rvc"] = (
    (funnel["has_viewed"] == 1) &
    (funnel["has_completed_after_view"] == 1)
).astype(int)

# 기존 코드 호환용
# 기존 converted_final 의미를 유지하려면 received -> completed로 둠
funnel["converted_final"] = funnel["converted_rc"]

# 상태 분류 (파생 컬럼): 최성영
funnel["status"] = np.select(
    [
        funnel["has_completed_rc"] == 1,
        funnel["has_viewed"] == 0,
        (funnel["has_viewed"] == 1) & (funnel["has_completed_rc"] == 0)
    ],
    [
        "converted",
        "not_viewed",
        "viewed_not_converted"
    ],
    default="other"
)

# 파생컬럼 분석

- 원본 유지
    - customer_id
    - offer_id
    - time_received
    - offer_type
    - offer_reward
    - difficulty
    - duration
    - web
    - email
    - mobile
    - social

- 퍼널 구성
    - offer_end_time
    - time_viewed
    - time_completed_rc
    - time_completed_vc

- 시간차
    - rv_time_diff
    - rc_time_diff
    - vc_time_diff

- 퍼널 플래그
    - has_viewed
    - has_completed_rc
    - has_completed_after_view

- 전환 플래그
    - converted_rv
    - converted_rc
    - converted_vc
    - converted_rvc
    - converted_final

- 상태
    - status

- 채널 파생
    - channel_count
    - channel_combo

- 세그먼트 파생
    - gender_group
    - age_group
    - income_group
    - member_year
    - member_cohort

- 확장
    - days_as_member
    - days_as_member_group
    - offer_count
    - offer_count_group


- 고객 기준 CSV
    - customer_id
    - segment
    - before_count
    - before_amount
    - after_count
    - after_amount

- 이주성: `received` 기준 메인 funnel 테이블 구조 통합, `gender_group`, `member_cohort` 반영
- 이승복: 퍼널 전환 관련 파생컬럼 생성
- 김지륜: 채널 관련 파생컬럼 생성
- 조준익: `age_group`, `income_group`, `member_year`, `days_as_member`, `days_as_member_group`, `offer_count`, `offer_count_group` 기준 제공
- 최성영: 고객 기준 전환 세그먼트 및 오퍼 전후 구매 비교 컬럼 생성

| 조원  | 담당 영역                              | 주요 파생컬럼                                                                                                                                                                                                    |
| --- | ---------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| 이주성 | 메인 funnel 구조 통합, 일부 세그먼트 컬럼 반영     | `offer_end_time`, `time_viewed`, `time_completed_rc`, `time_completed_vc`, `gender_group`, `member_cohort`                                                                                                 |
| 이승복 | 퍼널 전환 파생컬럼                         | `rv_time_diff`, `rc_time_diff`, `vc_time_diff`, `has_viewed`, `has_completed_rc`, `has_completed_after_view`, `converted_rv`, `converted_rc`, `converted_vc`, `converted_rvc`, `converted_final`, `status` |
| 김지륜 | 채널 파생컬럼                            | `channel_count`, `channel_combo`                                                                                                                                                                           |
| 조준익 | 고객 세그먼트 / 가입기간 / 오퍼 수신량 파생컬럼 기준 제공 | `age_group`, `income_group`, `member_year`, `days_as_member`, `days_as_member_group`, `offer_count`, `offer_count_group`                                                                                   |
| 최성영 | 고객 기준 비교 컬럼                        | `segment`, `before_count`, `before_amount`, `after_count`, `after_amount`                                                                                                                                  |



발견한 주의사항
1. 전환 기준은
    1. 조준익: seq 기반
    2. 이주성/이승복/김지륜: 가장 최근 유효 base 매칭
결론 =  base + duration 검증 방식 사용

1. received 기준 메인 funnel 테이블 (이주성)
2. channel_count, channel_combo (김지륜)
3. 고객 세그먼트 파생컬럼
    - gender_group, member_cohort (이주성)
    - age_group, income_group, member_year (조준익)
4. days_as_member, days_as_member_group, offer_count, offer_count_group (조준익)
5. segment, before_count, before_amount, after_count, after_amount (최성영, 고객 기준 별도 테이블)

# 김지륜

In [108]:
# ============================================================
# 채널 파생컬럼 생성: 김지륜
# - channel_count : 발송 채널 수
# - channel_combo : 발송 채널 조합
# ============================================================
channel_cols = ['web', 'email', 'mobile', 'social']

# 채널 컬럼 정리
funnel[channel_cols] = funnel[channel_cols].fillna(0).astype(int)

# 채널 수
funnel['channel_count'] = funnel[channel_cols].sum(axis=1)

# 채널 조합
funnel['channel_combo'] = funnel[channel_cols].apply(
    lambda row: ' + '.join([col for col in channel_cols if row[col] == 1]) if row.sum() > 0 else 'none',
    axis=1
)

print("channel_count / channel_combo 생성 완료")
display(
    funnel[
        ['web', 'email', 'mobile', 'social', 'channel_count', 'channel_combo']
    ].drop_duplicates().sort_values(['channel_count', 'channel_combo'])
)

channel_count / channel_combo 생성 완료


,web,email,mobile,social,channel_count,channel_combo
7,1,1,0,0,2,web + email
2,0,1,1,1,3,email + mobile + social
0,1,1,1,0,3,web + email + mobile
3,1,1,1,1,4,web + email + mobile + social


# 조준익+이주성

In [ ]:
# ============================================================
# 고객 세그먼트 파생컬럼 생성: 조준익 + 이주성
# ============================================================

# 1) 성별 그룹
funnel['gender_group'] = funnel['gender'].fillna('Missing')

# 2) 가입일 날짜형 변환
# - 조준익 segment2는 datetime 변환 후 year를 뽑는 방식
became_member_str = funnel['became_member_on'].astype(str).str.replace('-', '', regex=False)

funnel['became_member_on_dt'] = pd.to_datetime(
    became_member_str,
    format='%Y%m%d',
    errors='coerce'
)

# 3) 가입 연도
# - 조준익 파일의 member_year 흐름 반영
funnel['member_year'] = funnel['became_member_on_dt'].dt.year.astype('Int64')

# 4) 소득 구간
# - 조준익 segment2 기준 반영
funnel['income_group'] = pd.cut(
    funnel['income'],
    bins=[0, 40000, 60000, 80000, 100000, 200000],
    labels=['~4만', '4~6만', '6~8만', '8~10만', '10만+']
).astype('object').fillna('Missing')

# 5) 나이 구간
# - 조준익 segment2 기준 반영
funnel['age_group'] = pd.cut(
    funnel['age'],
    bins=[0, 30, 40, 50, 60, 120],
    labels=['~30대', '40대', '50대', '60대', '70대+']
).astype('object').fillna('Missing')

# 6) 가입 코호트
# - 조준익 파일에는 없지만, 통합 대시보드용 확장 컬럼으로 유지
funnel['member_cohort'] = pd.cut(
    funnel['member_year'],
    bins=[2013, 2015, 2017, 2019],
    right=False,
    labels=['2013-2014', '2015-2016', '2017-2018']
).astype('object').fillna('Missing')

print("세그먼트 파생컬럼 생성 완료")
display(
    funnel[
        [
            'gender', 'gender_group',
            'age', 'age_group',
            'income', 'income_group',
            'became_member_on', 'became_member_on_dt', 'member_year', 'member_cohort'
        ]
    ].head()
)

세그먼트 파생컬럼 생성 완료


,gender,gender_group,age,age_group,income,income_group,became_member_on,became_member_on_dt,member_year,member_cohort
0,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
1,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
2,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
3,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018
4,M,M,33.0,40대,72000.0,6~8만,2017-04-21,2017-04-21,2017,2017-2018


In [ ]:
# ============================================================
# 가입 경과일 / 오퍼 수신 횟수 파생컬럼 생성: 조준익
# ============================================================

# 1) 가입 경과일 계산 기준일
last_date = pd.Timestamp('2018-08-01')

# 2) 가입 경과일
funnel['days_as_member'] = (
    last_date - funnel['became_member_on_dt']
).dt.days


# 3) 가입 경과일 구간화
funnel['days_as_member_group'] = pd.cut(
    funnel['days_as_member'],
    bins=[0, 200, 500, 1000, 2000],
    labels=['~200일', '~500일', '~1000일', '1000일+']
).astype('object').fillna('Missing')

# 4) 고객별 오퍼 수신 횟수
offer_count_df = (
    funnel.groupby('customer_id')['offer_id']
    .count()
    .reset_index(name='offer_count')
)

funnel = funnel.merge(
    offer_count_df,
    on='customer_id',
    how='left'
)

# 5) 오퍼 수신 횟수 구간화
funnel['offer_count_group'] = pd.cut(
    funnel['offer_count'],
    bins=[0, 3, 6, 10, 20, 100],
    labels=['1~3', '4~6', '7~10', '11~20', '21+']
).astype('object').fillna('Missing')

# 6) 히트맵 재현용 그룹
funnel['offer_count_group_heatmap'] = pd.cut(
    funnel['offer_count'],
    bins=[0, 5, 10, 20, 100],
    labels=['~5회', '~10회', '~20회', '20회+']
).astype('object').fillna('Missing')

print("days_as_member / days_as_member_group / offer_count / offer_count_group 생성 완료")

display(
    funnel[
        [
            'customer_id',
            'became_member_on_dt',
            'days_as_member',
            'days_as_member_group',
            'offer_count',
            'offer_count_group',
            'offer_count_group_heatmap'
        ]
    ].head()
)

days_as_member / days_as_member_group / offer_count / offer_count_group 생성 완료


,customer_id,became_member_on_dt,days_as_member,days_as_member_group,offer_count,offer_count_group,offer_count_group_heatmap
0,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5,4~6,~5회
1,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5,4~6,~5회
2,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5,4~6,~5회
3,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5,4~6,~5회
4,0009655768c64bdeb2e877511632db8f,2017-04-21,467.0,~500일,5,4~6,~5회


# 최성영

A. 비전환 상태분류

B. offer 전후 비교

In [111]:
# ============================================================
# 고객 기준 전환 세그먼트 + 오퍼 전후 구매 비교 테이블 생성: 최성영
# - 최성영 원본 의도 반영:
#   전환된 offer(converted_final=1) 기준으로
#   received 전후 7일 transaction을 고객별 집계
# ============================================================

# ------------------------------------------------------------
# 1. 고객 기준 segment 생성
# - 고객이 한 번이라도 전환(converted_final=1)했으면 'converted'
# - 한 번도 전환하지 않았으면 'not_converted'
# ------------------------------------------------------------
customer_segment = (
    funnel.groupby('customer_id', as_index=False)['converted_final']
    .max()
    .rename(columns={'converted_final': 'converted'})
)

customer_segment['segment'] = customer_segment['converted'].map({
    1: 'converted',
    0: 'not_converted'
})

# ------------------------------------------------------------
# 2. transaction 이벤트만 분리
# ------------------------------------------------------------
transactions = (
    df2[df2['event'] == 'transaction'][['customer_id', 'time', 'amount']]
    .copy()
    .rename(columns={'time': 'time_transaction'})
)

# amount 숫자형 변환
transactions['amount'] = pd.to_numeric(
    transactions['amount'],
    errors='coerce'
).fillna(0)

print("transaction 행 수:", len(transactions))
display(transactions.head())

# ------------------------------------------------------------
# 3. 전환된 offer 기준 received 시점만 비교 기준으로 사용
# - 최성영 원본의 converted_rc 의도 반영
# ------------------------------------------------------------
compare_base = funnel.loc[
    funnel['converted_final'] == 1,
    ['customer_id', 'time_received']
].copy()

print("전환된 offer 기준 compare_base 행 수:", len(compare_base))
display(compare_base.head())

# ------------------------------------------------------------
# 4. 고객 기준으로 transaction 결합 후
#    received 전 7일 / 후 7일 플래그 생성
# ------------------------------------------------------------
compare_merged = compare_base.merge(
    transactions,
    on='customer_id',
    how='left'
)

window_hours = 7 * 24

compare_merged['before_flag'] = (
    (compare_merged['time_transaction'] >= compare_merged['time_received'] - window_hours) &
    (compare_merged['time_transaction'] < compare_merged['time_received'])
)

compare_merged['after_flag'] = (
    (compare_merged['time_transaction'] >= compare_merged['time_received']) &
    (compare_merged['time_transaction'] <= compare_merged['time_received'] + window_hours)
)

display(compare_merged.head())

# ------------------------------------------------------------
# 5. 고객 기준 구매 횟수 / 구매 금액 집계
# ------------------------------------------------------------
before_df = compare_merged[compare_merged['before_flag']].copy()
after_df = compare_merged[compare_merged['after_flag']].copy()

before_agg = (
    before_df.groupby('customer_id', as_index=False)
    .agg(
        before_count=('amount', 'count'),
        before_amount=('amount', 'sum')
    )
)

after_agg = (
    after_df.groupby('customer_id', as_index=False)
    .agg(
        after_count=('amount', 'count'),
        after_amount=('amount', 'sum')
    )
)

# ------------------------------------------------------------
# 6. 고객 기준 최종 테이블 결합
# - 고객 segment + before/after 집계 결합
# ------------------------------------------------------------
customer_compare = customer_segment.merge(
    before_agg,
    on='customer_id',
    how='left'
).merge(
    after_agg,
    on='customer_id',
    how='left'
)

# 결측값 처리
for col in ['before_count', 'before_amount', 'after_count', 'after_amount']:
    customer_compare[col] = customer_compare[col].fillna(0)

customer_compare['before_count'] = customer_compare['before_count'].astype(int)
customer_compare['after_count'] = customer_compare['after_count'].astype(int)

print("customer_compare 생성 완료")
print("행 수:", len(customer_compare))
display(customer_compare.head())

transaction 행 수: 138953


,customer_id,time_transaction,amount
12654,02c083884c7d45b39cc68e1314fec56c,0,0.83
12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,0,34.56
12659,54890f68699049c2a04d415abc25e717,0,13.23
12670,b2f1cd155b864803ad8334cdf13c4bd2,0,19.51
12671,fe97aa22dd3e48c8b143116a8403dd52,0,18.97


전환된 offer 기준 compare_base 행 수: 33152


,customer_id,time_received
0,0009655768c64bdeb2e877511632db8f,576
3,0009655768c64bdeb2e877511632db8f,408
4,0009655768c64bdeb2e877511632db8f,504
7,0011e0d4e6b944f998e987f904e8c1e5,408
8,0011e0d4e6b944f998e987f904e8c1e5,168


,customer_id,time_received,time_transaction,amount,before_flag,after_flag
0,0009655768c64bdeb2e877511632db8f,576,228,22.16,False,False
1,0009655768c64bdeb2e877511632db8f,576,414,8.57,True,False
2,0009655768c64bdeb2e877511632db8f,576,528,14.11,True,False
3,0009655768c64bdeb2e877511632db8f,576,552,13.56,True,False
4,0009655768c64bdeb2e877511632db8f,576,576,10.27,False,True


customer_compare 생성 완료
행 수: 16994


,customer_id,converted,segment,before_count,before_amount,after_count,after_amount
0,0009655768c64bdeb2e877511632db8f,1,converted,4,44.81,12,166.01
1,00116118485d4dfda04fdbaba9a87b5c,0,not_converted,0,0.00,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,1,converted,2,25.42,5,88.02
3,0020c2b971eb4e9188eac86d93036a77,1,converted,0,0.00,6,149.43
4,0020ccbbb6d84e358d3414a3ff76cffd,1,converted,10,136.26,11,137.78


In [112]:
# 원본처럼 쓸꺼면 아래의 코드 사용
# ------------------------------------------------------------
# 6. 고객 기준 최종 비교 테이블 결합
# ------------------------------------------------------------
# customer_compare = (
#     before_agg.set_index('customer_id')
#     .join(
#         after_agg.set_index('customer_id'),
#         how='outer'
#     )
#     .fillna(0)
#     .reset_index()
# )

# customer_compare['before_count'] = customer_compare['before_count'].astype(int)
# customer_compare['after_count'] = customer_compare['after_count'].astype(int)

# print("customer_compare 생성 완료")
# print("행 수:", len(customer_compare))
# display(customer_compare.head())


# customer_compare_final = customer_compare[
#     [
#         'customer_id',
#         'segment',
#         'before_count',
#         'before_amount',
#         'after_count',
#         'after_amount'
#     ]
# ].copy()

# 최종

## funnel 최종 컬럼 정리

In [113]:
# ============================================================
# 메인 funnel 최종 저장용 컬럼 정리
# ============================================================
funnel_final = funnel[
    [
        # 원본 식별 / 오퍼 속성
        'customer_id', 'offer_id', 'time_received',
        'offer_type', 'offer_reward', 'difficulty', 'duration',

        # 채널 원본
        'web', 'email', 'mobile', 'social',

        # 퍼널 시간
        'offer_end_time', 'time_viewed', 'time_completed_rc', 'time_completed_vc',

        # 시간차
        'rv_time_diff', 'rc_time_diff', 'vc_time_diff',

        # 퍼널 플래그
        'has_viewed', 'has_completed_rc', 'has_completed_after_view',

        # 전환 플래그
        'converted_rv', 'converted_rc', 'converted_vc', 'converted_rvc', 'converted_final',

        # 상태
        'status',

        # 채널 파생
        'channel_count', 'channel_combo',

        # 세그먼트 파생
        'has_profile', 'gender_group', 'age_group', 
        'income_group', 'member_year', 'member_cohort',

        # 확장 파생
        'days_as_member', 'days_as_member_group', 'offer_count', 'offer_count_group'
        
    ]
].copy()

print("funnel_final 생성 완료")
print("행 수:", len(funnel_final))
print("열 수:", funnel_final.shape[1])
display(funnel_final.head())

funnel_final 생성 완료
행 수: 76277
열 수: 39


,customer_id,offer_id,time_received,offer_type,offer_reward,difficulty,duration,web,email,mobile,social,offer_end_time,time_viewed,time_completed_rc,time_completed_vc,rv_time_diff,rc_time_diff,vc_time_diff,has_viewed,has_completed_rc,has_completed_after_view,converted_rv,converted_rc,converted_vc,converted_rvc,converted_final,status,channel_count,channel_combo,has_profile,gender_group,age_group,income_group,member_year,member_cohort,days_as_member,days_as_member_group,offer_count,offer_count_group
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576,discount,2.0,10.0,7.0,1,1,1,0,744.0,NaN,576.0,NaN,NaN,0.0,NaN,0,1,0,0,1,0,0,1,converted,3,web + email + mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5,4~6
1,0009655768c64bdeb2e877511632db8f,3f207df678b143eea3cee63160fa8bed,336,informational,0.0,0.0,4.0,1,1,1,0,432.0,372.0,NaN,NaN,36.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,web + email + mobile,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5,4~6
2,0009655768c64bdeb2e877511632db8f,5a8bc65990b245e5a138643cd4eb9837,168,informational,0.0,0.0,3.0,0,1,1,1,240.0,192.0,NaN,NaN,24.0,NaN,NaN,1,0,0,1,0,0,0,0,viewed_not_converted,3,email + mobile + social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5,4~6
3,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,408,bogo,5.0,5.0,5.0,1,1,1,1,528.0,456.0,414.0,NaN,48.0,6.0,NaN,1,1,0,1,1,0,0,1,converted,4,web + email + mobile + social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5,4~6
4,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,504,discount,2.0,10.0,10.0,1,1,1,1,744.0,540.0,528.0,NaN,36.0,24.0,NaN,1,1,0,1,1,0,0,1,converted,4,web + email + mobile + social,1,M,40대,6~8만,2017,2017-2018,467.0,~500일,5,4~6


In [114]:
# ============================================================
# 고객 기준 최종 저장용 컬럼만 정리
# ============================================================
customer_compare_final = customer_compare[
    [
        'customer_id',
        'segment',
        'before_count',
        'before_amount',
        'after_count',
        'after_amount'
    ]
].copy()

print("customer_compare_final 생성 완료")
display(customer_compare_final.head())

customer_compare_final 생성 완료


,customer_id,segment,before_count,before_amount,after_count,after_amount
0,0009655768c64bdeb2e877511632db8f,converted,4,44.81,12,166.01
1,00116118485d4dfda04fdbaba9a87b5c,not_converted,0,0.00,0,0.00
2,0011e0d4e6b944f998e987f904e8c1e5,converted,2,25.42,5,88.02
3,0020c2b971eb4e9188eac86d93036a77,converted,0,0.00,6,149.43
4,0020ccbbb6d84e358d3414a3ff76cffd,converted,10,136.26,11,137.78


# 작업 진행

# 최성영
1. 상태별 개체수 + 비율
2. 퍼널 구조
3. offer_type별 상태 분포
4. 채널별 상태 분포
5. 전/후 평균 비교(±7일 기준)

# 조준익

# 점검

In [154]:
# ============================================================
# 최종 저장 전 점검
# ============================================================
print("=" * 80)
print("funnel_final 정보")
print("=" * 80)
print("행/열:", funnel_final.shape)
print()
print(funnel_final.columns.tolist())
print()

print("=" * 80)
print("customer_compare_final 정보")
print("=" * 80)
print("행/열:", customer_compare_final.shape)
print()
print(customer_compare_final.columns.tolist())

funnel_final 정보
행/열: (76277, 39)

['customer_id', 'offer_id', 'time_received', 'offer_type', 'offer_reward', 'difficulty', 'duration', 'web', 'email', 'mobile', 'social', 'offer_end_time', 'time_viewed', 'time_completed_rc', 'time_completed_vc', 'rv_time_diff', 'rc_time_diff', 'vc_time_diff', 'has_viewed', 'has_completed_rc', 'has_completed_after_view', 'converted_rv', 'converted_rc', 'converted_vc', 'converted_rvc', 'converted_final', 'status', 'channel_count', 'channel_combo', 'has_profile', 'gender_group', 'age_group', 'income_group', 'member_year', 'member_cohort', 'days_as_member', 'days_as_member_group', 'offer_count', 'offer_count_group']

customer_compare_final 정보
행/열: (16994, 6)

['customer_id', 'segment', 'before_count', 'before_amount', 'after_count', 'after_amount']


In [ ]:
# # ============================================================
# # 최종 CSV 저장
# # ============================================================
# funnel_final.to_csv('funnel_final.csv', index=False, encoding='utf-8-sig')
# customer_compare_final.to_csv('customer_compare_final.csv', index=False, encoding='utf-8-sig')

# print("CSV 저장 완료")
# print("- funnel_final.csv")
# print("- customer_compare_final.csv")

| 최종 통합 컬럼명                  | 조원/원본에서 보인 이름                                  | 담당/출처              | 의미                                                           |
| -------------------------- | ---------------------------------------------- | ------------------ | ------------------------------------------------------------ |
| `offer_end_time`           | `offer_end_time`                               | 이주성, 이승복           | 오퍼 종료 시점 (`time_received + duration*24`)                     |
| `time_viewed`              | `time_viewed`                                  | 이주성, 이승복           | received에 연결된 viewed 시점                                      |
| `time_completed_rc`        | `time_completed`, `time_completed_rc`          | 김지륜, 이승복, 이주성      | received 기준 completed 시점                                     |
| `time_completed_vc`        | `time_completed_vc`                            | 이승복, 이주성           | viewed 이후 completed 시점                                       |
| `rv_time_diff`             | `rv_time_diff`                                 | 이승복, 이주성           | received → viewed 시간차                                        |
| `rc_time_diff`             | `time_diff`, `rc_time_diff`                    | 김지륜, 이승복, 이주성      | received → completed 시간차                                     |
| `vc_time_diff`             | `vc_time_diff`                                 | 이승복, 이주성           | viewed → completed 시간차                                       |
| `has_viewed`               | `has_viewed`, `has_veiwed`                     | 이승복, 최성영           | viewed 발생 여부                                                 |
| `has_completed_rc`         | `has_completed`, `has_completed_rc`            | 김지륜, 이승복, 이주성      | received 이후 completed 여부                                     |
| `has_completed_after_view` | `has_completed_after_view`, `has_completed_vc` | 이승복, 이주성           | viewed 이후 completed 여부                                       |
| `converted_rv`             | `converted_rv`                                 | 이승복, 이주성           | received 대비 viewed 전환 여부                                     |
| `converted_rc`             | `converted_final`, `converted_rc`              | 김지륜, 이승복, 이주성, 최성영 | received 대비 completed 전환 여부                                  |
| `converted_vc`             | `converted_vc`                                 | 이승복, 이주성           | viewed 대비 completed 전환 여부                                    |
| `converted_rvc`            | `converted_rvc`                                | 이승복, 이주성           | viewed도 했고 completed도 했는지                                    |
| `converted_final`          | `converted_final`, `is_converted`, `converted` | 김지륜, 이승복, 조준익 일부   | 최종 대표 전환 여부                                                  |
| `status`                   | `status`                                       | 이승복, 최성영           | 오퍼 상태 분류 (`converted`, `not_viewed`, `viewed_not_converted`) |
| `channel_count`            | `channel_count`                                | 김지륜                | 발송 채널 수                                                      |
| `channel_combo`            | `channel_combo`                                | 김지륜                | 발송 채널 조합 문자열                                                 |
| `has_profile`              | `has_profile`, `profile_missing`               | 조준익                | 프로필 정보 보유 여부                                                 |
| `gender_group`             | `gender_group`                                 | 이주성                | 성별 그룹                                                        |
| `age_group`                | `age_group`                                    | 이주성, 조준익           | 나이 구간                                                        |
| `income_group`             | `income_group`                                 | 이주성, 조준익           | 소득 구간                                                        |
| `member_year`              | `member_year`, `membership_year`               | 이주성, 조준익           | 가입 연도                                                        |
| `member_cohort`            | `member_cohort`                                | 이주성                | 가입 코호트 구간                                                    |
| `days_as_member`           | `days_as_member`, `member_tenure`              | 조준익                | 가입 후 경과일                                                     |
| `days_as_member_group`     | `days_as_member_group`, `tenure_group`         | 조준익                | 가입 경과일 구간                                                    |
| `offer_count`              | `offer_count`                                  | 조준익                | 고객별 오퍼 수신 횟수                                                 |
| `offer_count_group`        | `offer_count_group`                            | 조준익                | 오퍼 수신 횟수 구간                                                  |

| 최종 통합 컬럼명       | 조원/원본에서 보인 이름                          | 담당/출처       | 의미                |
| --------------- | -------------------------------------- | ----------- | ----------------- |
| `segment`       | `segment`, `converted / not_converted` | 최성영, 조준익 일부 | 고객 기준 전환 세그먼트     |
| `before_count`  | `before_count`                         | 최성영         | 오퍼 기준 이전 기간 구매 횟수 |
| `before_amount` | `before_amount`                        | 최성영         | 오퍼 기준 이전 기간 구매 금액 |
| `after_count`   | `after_count`                          | 최성영         | 오퍼 기준 이후 기간 구매 횟수 |
| `after_amount`  | `after_amount`                         | 최성영         | 오퍼 기준 이후 기간 구매 금액 |